# BikeToDrive Datawarehouse ETL

This notebook implements the **Data Warehouse** for BikeToDrive using the five supplied source databases. We create a star schema in a new SQLite database (`dwh.db`) and load the dimension and fact tables. Two dimension tables (`dim_klant` and `dim_monteur`) are implemented as Slowly Changing Dimension (SCD) Type 2, while all other dimensions follow SCD Type 1 semantics.

The overall steps are:

1. **Inspect source databases** – open the five source SQLite files and examine their table structures.
2. **Define the data warehouse schema** – create dimension and fact tables with appropriate surrogate keys, datatypes, and relationships.
3. **Load dimension tables** – extract distinct rows from source tables, detect new and changed records, and apply the SCD rules (type 1 or type 2).
4. **Load fact tables** – transform each source fact table and map business keys to surrogate keys in the corresponding dimension tables.
5. **Test SCD behaviour** – modify records in the source to demonstrate how SCD Type 1 and Type 2 updates are handled.

The current date used for SCD timestamps is set to `2026-03-23` to align with the assignment's timeline.


In [7]:
import sqlite3
import pandas as pd
from datetime import datetime

path_accessoire_verkoop = 'BikeToDrive_1_Accessoireverkoop.db'
path_fiets_verkoop      = 'BikeToDrive_2_Fietsverkoop.db'
path_onderhoud          = 'BikeToDrive_3_Onderhoud.db'
path_accessoire_inkoop  = 'BikeToDrive_4_Accessoire_Inkoop.db'
path_fiets_inkoop       = 'BikeToDrive_5_Fiets_Inkoop.db'

# Data warehouse
dwh_path = 'dwh.db'

# Date for SCD
today = datetime(2026, 3, 23)


In [8]:
conn_dwh = sqlite3.connect(dwh_path)
cur = conn_dwh.cursor()

for t in ['dim_klant','dim_monteur','dim_filiaal','dim_accessoire','dim_fiets','dim_fabrikant','dim_leverancier','fact_accessoire_verkoop','fact_fiets_verkoop','fact_onderhoud','fact_accessoire_inkoop','fact_fiets_inkoop']:
    cur.execute(f"DROP TABLE IF EXISTS {t}")

cur.execute('''CREATE TABLE dim_klant (
    klant_sk      INTEGER PRIMARY KEY AUTOINCREMENT,
    klantnr       INTEGER,
    naam          TEXT,
    adres         TEXT,
    woonplaats    TEXT,
    geslacht      TEXT,
    geboortedatum DATE,
    start_date    DATE,
    end_date      DATE,
    is_current    INTEGER
)''')

cur.execute('''CREATE TABLE dim_monteur (
    monteur_sk    INTEGER PRIMARY KEY AUTOINCREMENT,
    monteurnr     INTEGER,
    naam          TEXT,
    woonplaats    TEXT,
    uurloon       REAL,
    filiaal       INTEGER,
    start_date    DATE,
    end_date      DATE,
    is_current    INTEGER
)''')

cur.execute('''CREATE TABLE dim_filiaal (
    filiaal_sk    INTEGER PRIMARY KEY AUTOINCREMENT,
    filiaalnr     INTEGER,
    naam          TEXT,
    adres         TEXT,
    provincie     TEXT
)''')

cur.execute('''CREATE TABLE dim_accessoire (
    accessoire_sk  INTEGER PRIMARY KEY AUTOINCREMENT,
    accessoirenr   INTEGER,
    naam           TEXT,
    standaardprijs REAL,
    inkoopprijs    REAL,
    soort          TEXT,
    leverancier    INTEGER
)''')

cur.execute('''CREATE TABLE dim_fiets (
    fiets_sk       INTEGER PRIMARY KEY AUTOINCREMENT,
    fietsnr        INTEGER,
    soort          TEXT,
    merk           TEXT,
    type           TEXT,
    standaardprijs REAL,
    inkoopprijs    REAL,
    kleur          TEXT,
    fabrikant      INTEGER
)''')

cur.execute('''CREATE TABLE dim_fabrikant (
    fabrikant_sk  INTEGER PRIMARY KEY AUTOINCREMENT,
    fabrikantnr   INTEGER,
    naam          TEXT,
    adres         TEXT,
    plaats        TEXT
)''')

cur.execute('''CREATE TABLE dim_leverancier (
    leverancier_sk INTEGER PRIMARY KEY AUTOINCREMENT,
    leveranciernr  INTEGER,
    naam           TEXT,
    adres          TEXT,
    woonplaats     TEXT
)''')

cur.execute('''CREATE TABLE fact_accessoire_verkoop (
    verkoop_sk           INTEGER PRIMARY KEY AUTOINCREMENT,
    accessoire_verkoopnr INTEGER,
    datum                DATE,
    aantal               INTEGER,
    verkoopprijs         REAL,
    klant_sk             INTEGER,
    monteur_sk           INTEGER,
    accessoire_sk        INTEGER,
    FOREIGN KEY (klant_sk) REFERENCES dim_klant(klant_sk),
    FOREIGN KEY (monteur_sk) REFERENCES dim_monteur(monteur_sk),
    FOREIGN KEY (accessoire_sk) REFERENCES dim_accessoire(accessoire_sk)
)''')

cur.execute('''CREATE TABLE fact_fiets_verkoop (
    verkoop_sk        INTEGER PRIMARY KEY AUTOINCREMENT,
    fiets_verkoopnr   INTEGER,
    datum             DATE,
    aantal            INTEGER,
    verkoopprijs      REAL,
    klant_sk          INTEGER,
    monteur_sk        INTEGER,
    fiets_sk          INTEGER,
    FOREIGN KEY (klant_sk) REFERENCES dim_klant(klant_sk),
    FOREIGN KEY (monteur_sk) REFERENCES dim_monteur(monteur_sk),
    FOREIGN KEY (fiets_sk) REFERENCES dim_fiets(fiets_sk)
)''')

cur.execute('''CREATE TABLE fact_onderhoud (
    onderhoud_sk  INTEGER PRIMARY KEY AUTOINCREMENT,
    onderhoudnr   INTEGER,
    datum         DATE,
    starttijd     TEXT,
    eindtijd      TEXT,
    monteur_sk    INTEGER,
    fiets_sk      INTEGER,
    FOREIGN KEY (monteur_sk) REFERENCES dim_monteur(monteur_sk),
    FOREIGN KEY (fiets_sk) REFERENCES dim_fiets(fiets_sk)
)''')

cur.execute('''CREATE TABLE fact_accessoire_inkoop (
    inkoop_sk     INTEGER PRIMARY KEY AUTOINCREMENT,
    inkoopnr      INTEGER,
    inkoopmaand   INTEGER,
    inkoopjaar    INTEGER,
    aantal        INTEGER,
    accessoire_sk INTEGER,
    FOREIGN KEY (accessoire_sk) REFERENCES dim_accessoire(accessoire_sk)
)''')

cur.execute('''CREATE TABLE fact_fiets_inkoop (
    inkoop_sk     INTEGER PRIMARY KEY AUTOINCREMENT,
    inkoopnr      INTEGER,
    inkoopmaand   INTEGER,
    inkoopjaar    INTEGER,
    aantal        INTEGER,
    fiets_sk      INTEGER,
    FOREIGN KEY (fiets_sk) REFERENCES dim_fiets(fiets_sk)
)''')

conn_dwh.commit()


In [9]:
def load_dimension_type1(conn_source, source_query, dwh_conn, dwh_table, key_column, columns):
    src_df = pd.read_sql_query(source_query, conn_source)
    tgt_df = pd.read_sql_query(f"SELECT * FROM {dwh_table}", dwh_conn)
    for _, row in src_df.iterrows():
        existing = tgt_df[tgt_df[key_column] == row[key_column]]
        if existing.empty:
            cols = ', '.join([key_column] + columns)
            placeholders = ', '.join(['?' for _ in range(len(columns)+1)])
            values = [row[key_column]] + [row[col] for col in columns]
            dwh_conn.execute(f"INSERT INTO {dwh_table} ({cols}) VALUES ({placeholders})", values)
        else:
            possible_sk = [c for c in tgt_df.columns if c.endswith('_sk')]
            sk_col = possible_sk[0]
            sk = int(existing.iloc[0][sk_col])
            set_clause = ', '.join([f"{col} = ?" for col in columns])
            values = [row[col] for col in columns] + [sk]
            dwh_conn.execute(f"UPDATE {dwh_table} SET {set_clause} WHERE {sk_col} = ?", values)
    dwh_conn.commit()


def load_dimension_type2(conn_source, source_query, dwh_conn, dwh_table, key_column, columns):
    src_df = pd.read_sql_query(source_query, conn_source)
    tgt_df = pd.read_sql_query(f"SELECT * FROM {dwh_table}", dwh_conn)
    for _, row in src_df.iterrows():
        existing_current = tgt_df[(tgt_df[key_column] == row[key_column]) & (tgt_df['is_current'] == 1)]
        attrs = {col: row[col] for col in columns}
        if existing_current.empty:
            insert_values = [row[key_column]] + list(attrs.values()) + [today.date(), None, 1]
            placeholders = ', '.join(['?' for _ in insert_values])
            cols = ', '.join([key_column] + columns + ['start_date','end_date','is_current'])
            dwh_conn.execute(f"INSERT INTO {dwh_table} ({cols}) VALUES ({placeholders})", insert_values)
        else:
            current_row = existing_current.iloc[0]
            changed = False
            for col in columns:
                if str(current_row[col]) != str(attrs[col]):
                    changed = True
                    break
            if changed:
                possible_sk = [c for c in tgt_df.columns if c.endswith('_sk')]
                sk_col = possible_sk[0]
                current_sk = int(current_row[sk_col])
                dwh_conn.execute(
                    f"UPDATE {dwh_table} SET end_date = ?, is_current = 0 WHERE {sk_col} = ?",
                    (today.date(), current_sk)
                )
                insert_values = [row[key_column]] + list(attrs.values()) + [today.date(), None, 1]
                placeholders = ', '.join(['?' for _ in insert_values])
                cols = ', '.join([key_column] + columns + ['start_date','end_date','is_current'])
                dwh_conn.execute(f"INSERT INTO {dwh_table} ({cols}) VALUES ({placeholders})", insert_values)
    dwh_conn.commit()


In [10]:
# Open source connections
conn_acc_verkoop = sqlite3.connect(path_accessoire_verkoop)
conn_fiets_verkoop = sqlite3.connect(path_fiets_verkoop)

# dim_klant
source_query_klant = "SELECT DISTINCT klantnr AS klantnr, naam, adres, woonplaats, geslacht, geboortedatum FROM Klant"
load_dimension_type2(conn_fiets_verkoop, source_query_klant, conn_dwh, 'dim_klant', 'klantnr', ['naam','adres','woonplaats','geslacht','geboortedatum'])

# dim_monteur
source_query_monteur = "SELECT DISTINCT monteurnr AS monteurnr, naam, woonplaats, uurloon, filiaal FROM Monteur"
load_dimension_type2(conn_fiets_verkoop, source_query_monteur, conn_dwh, 'dim_monteur', 'monteurnr', ['naam','woonplaats','uurloon','filiaal'])

# dim_filiaal
source_query_filiaal = "SELECT DISTINCT filiaalnr AS filiaalnr, naam, adres, provincie FROM Filiaal"
load_dimension_type1(conn_fiets_verkoop, source_query_filiaal, conn_dwh, 'dim_filiaal', 'filiaalnr', ['naam','adres','provincie'])

# dim_accessoire
source_query_accessoire = "SELECT DISTINCT accessoirenr AS accessoirenr, naam, standaardprijs, inkoopprijs, soort, leverancier FROM Accessoire"
load_dimension_type1(conn_acc_verkoop, source_query_accessoire, conn_dwh, 'dim_accessoire', 'accessoirenr', ['naam','standaardprijs','inkoopprijs','soort','leverancier'])

# dim_fiets
source_query_fiets = "SELECT DISTINCT fietsnr AS fietsnr, soort, merk, type, standaardprijs, inkoopprijs, kleur, fabrikant FROM Fiets"
load_dimension_type1(conn_fiets_verkoop, source_query_fiets, conn_dwh, 'dim_fiets', 'fietsnr', ['soort','merk','type','standaardprijs','inkoopprijs','kleur','fabrikant'])

# dim_fabrikant
source_query_fabrikant = "SELECT DISTINCT fabrikantnr AS fabrikantnr, naam, adres, plaats FROM Fabrikant"
load_dimension_type1(conn_fiets_verkoop, source_query_fabrikant, conn_dwh, 'dim_fabrikant', 'fabrikantnr', ['naam','adres','plaats'])

# dim_leverancier
source_query_leverancier = "SELECT DISTINCT leveranciernr AS leveranciernr, naam, adres, woonplaats FROM Leverancier"
load_dimension_type1(conn_acc_verkoop, source_query_leverancier, conn_dwh, 'dim_leverancier', 'leveranciernr', ['naam','adres','woonplaats'])

conn_acc_verkoop.close()
conn_fiets_verkoop.close()


C:\Users\Bayra\AppData\Local\Temp\ipykernel_11292\2887445164.py:31: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  dwh_conn.execute(f"INSERT INTO {dwh_table} ({cols}) VALUES ({placeholders})", insert_values)
C:\Users\Bayra\AppData\Local\Temp\ipykernel_11292\2887445164.py:31: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  dwh_conn.execute(f"INSERT INTO {dwh_table} ({cols}) VALUES ({placeholders})", insert_values)


In [11]:
# Helper to map business keys to surrogate keys
def get_surrogate_key(dwh_conn, dim_table, key_column, key_value):
    # derive surrogate key column from dim_table: remove 'dim_' prefix and append '_sk'
    sk_col = dim_table[4:] + '_sk'
    df = pd.read_sql_query(f"SELECT {sk_col} AS sk, {key_column} FROM {dim_table}", dwh_conn)
    row = df[df[key_column] == key_value]
    if row.empty:
        return None
    return int(row.iloc[0]['sk'])

# Override to ensure current version of Type 2 dimensions
def get_surrogate_key(dwh_conn, dim_table, key_column, key_value):
    sk_col = dim_table[4:] + '_sk'
    df = pd.read_sql_query(f"SELECT * FROM {dim_table}", dwh_conn)
    # For Type 2 dimensions (with is_current), select only current records
    if 'is_current' in df.columns:
        df = df[df['is_current'] == 1]
    row = df[df[key_column] == key_value]
    if row.empty:
        return None
    return int(row.iloc[0][sk_col])

# Open connections
conn_acc_verkoop = sqlite3.connect(path_accessoire_verkoop)
conn_fiets_verkoop = sqlite3.connect(path_fiets_verkoop)
conn_onderhoud = sqlite3.connect(path_onderhoud)
conn_acc_inkoop = sqlite3.connect(path_accessoire_inkoop)
conn_fiets_inkoop = sqlite3.connect(path_fiets_inkoop)

# Accessoire verkoop
acc_verkoop = pd.read_sql_query("SELECT * FROM Accessoire_Verkoop", conn_acc_verkoop)
for _, row in acc_verkoop.iterrows():
    klant_sk      = get_surrogate_key(conn_dwh, 'dim_klant', 'klantnr', row['klant'])
    monteur_sk    = get_surrogate_key(conn_dwh, 'dim_monteur', 'monteurnr', row['monteur'])
    accessoire_sk = get_surrogate_key(conn_dwh, 'dim_accessoire', 'accessoirenr', row['accessoire'])
    conn_dwh.execute("INSERT INTO fact_accessoire_verkoop (accessoire_verkoopnr, datum, aantal, verkoopprijs, klant_sk, monteur_sk, accessoire_sk) VALUES (?,?,?,?,?,?,?)",
                     (row['accessoire_verkoopnr'], row['datum'], row['aantal'], row['verkoopprijs'], klant_sk, monteur_sk, accessoire_sk))

# Fiets verkoop
fiets_verkoop = pd.read_sql_query("SELECT * FROM Fiets_Verkoop", conn_fiets_verkoop)
for _, row in fiets_verkoop.iterrows():
    klant_sk   = get_surrogate_key(conn_dwh, 'dim_klant', 'klantnr', row['klant'])
    monteur_sk = get_surrogate_key(conn_dwh, 'dim_monteur', 'monteurnr', row['monteur'])
    fiets_sk   = get_surrogate_key(conn_dwh, 'dim_fiets', 'fietsnr', row['fiets'])
    conn_dwh.execute("INSERT INTO fact_fiets_verkoop (fiets_verkoopnr, datum, aantal, verkoopprijs, klant_sk, monteur_sk, fiets_sk) VALUES (?,?,?,?,?,?,?)",
                     (row['fiets_verkoopnr'], row['datum'], row['aantal'], row['verkoopprijs'], klant_sk, monteur_sk, fiets_sk))

# Onderhoud
onderhoud = pd.read_sql_query("SELECT * FROM Onderhoud", conn_onderhoud)
for _, row in onderhoud.iterrows():
    monteur_sk = get_surrogate_key(conn_dwh, 'dim_monteur', 'monteurnr', row['monteur'])
    fiets_sk   = get_surrogate_key(conn_dwh, 'dim_fiets', 'fietsnr', row['fiets'])
    conn_dwh.execute("INSERT INTO fact_onderhoud (onderhoudnr, datum, starttijd, eindtijd, monteur_sk, fiets_sk) VALUES (?,?,?,?,?,?)",
                     (row['onderhoudnr'], row['datum'], row['starttijd'], row['eindtijd'], monteur_sk, fiets_sk))

# Accessoire inkoop
acc_inkoop = pd.read_sql_query("SELECT * FROM Accessoire_Inkoop", conn_acc_inkoop)
for _, row in acc_inkoop.iterrows():
    accessoire_sk = get_surrogate_key(conn_dwh, 'dim_accessoire', 'accessoirenr', row['accessoire'])
    conn_dwh.execute("INSERT INTO fact_accessoire_inkoop (inkoopnr, inkoopmaand, inkoopjaar, aantal, accessoire_sk) VALUES (?,?,?,?,?)",
                     (row['inkoopnr'], row['inkoopmaand'], row['inkoopjaar'], row['aantal'], accessoire_sk))

# Fiets inkoop
fiets_inkoop = pd.read_sql_query("SELECT * FROM Fiets_Inkoop", conn_fiets_inkoop)
for _, row in fiets_inkoop.iterrows():
    fiets_sk = get_surrogate_key(conn_dwh, 'dim_fiets', 'fietsnr', row['fiets'])
    conn_dwh.execute("INSERT INTO fact_fiets_inkoop (inkoopnr, inkoopmaand, inkoopjaar, aantal, fiets_sk) VALUES (?,?,?,?,?)",
                     (row['inkoopnr'], row['inkoopmaand'], row['inkoopjaar'], row['aantal'], fiets_sk))

conn_dwh.commit()

# Close source connections
conn_acc_verkoop.close()
conn_fiets_verkoop.close()
conn_onderhoud.close()
conn_acc_inkoop.close()
conn_fiets_inkoop.close()


In [12]:
# Modify records to demonstrate SCD behaviour
conn_fiets_verkoop_mod = sqlite3.connect(path_fiets_verkoop)
conn_acc_verkoop_mod = sqlite3.connect(path_accessoire_verkoop)

conn_fiets_verkoop_mod.execute("UPDATE Klant SET adres = 'Nieuweweg 99' WHERE klantnr = 1")
conn_fiets_verkoop_mod.execute("UPDATE Monteur SET woonplaats = 'Nieuwe Stad' WHERE monteurnr = 1")
conn_acc_verkoop_mod.execute("UPDATE Accessoire SET standaardprijs = standaardprijs + 5 WHERE accessoirenr = 1")

# Reload dimensions after modifications
load_dimension_type2(conn_fiets_verkoop_mod, source_query_klant, conn_dwh, 'dim_klant', 'klantnr', ['naam','adres','woonplaats','geslacht','geboortedatum'])
load_dimension_type2(conn_fiets_verkoop_mod, source_query_monteur, conn_dwh, 'dim_monteur', 'monteurnr', ['naam','woonplaats','uurloon','filiaal'])
load_dimension_type1(conn_acc_verkoop_mod, source_query_accessoire, conn_dwh, 'dim_accessoire', 'accessoirenr', ['naam','standaardprijs','inkoopprijs','soort','leverancier'])

print('dim_klant after modification:')
print(pd.read_sql_query("SELECT * FROM dim_klant ORDER BY klantnr, start_date", conn_dwh).head(10))

print('dim_monteur after modification:')
print(pd.read_sql_query("SELECT * FROM dim_monteur ORDER BY monteurnr, start_date", conn_dwh).head(10))

print('dim_accessoire after modification:')
print(pd.read_sql_query("SELECT * FROM dim_accessoire ORDER BY accessoirenr", conn_dwh).head(5))

conn_fiets_verkoop_mod.close()
conn_acc_verkoop_mod.close()


dim_klant after modification:
   klant_sk  klantnr            naam             adres woonplaats geslacht  \
0         1        1      Jan Jansen     Kerkstraat 12  Amsterdam        M   
1        26        1      Jan Jansen      Nieuweweg 99  Amsterdam        M   
2         2        2  Sophie de Boer       Lindelaan 8  Rotterdam        V   
3         3        3   Pieter Visser     Havenstraat 3   Den Haag        M   
4         4        4       Emma Smit      Boomgaard 22    Haarlem        V   
5         5        5      Tom Bakker    Stationsweg 44     Leiden        M   
6         6        6     Lisa Meijer     Dijkstraat 10    Zaandam        V   
7         7        7   Bart de Vries  Brouwersgracht 7      Delft        M   
8         8        8  Julia van Dijk     Plataanlaan 5      Hoorn        V   
9         9        9       Kevin Mol         Singel 99    Alkmaar        M   

  geboortedatum  start_date    end_date  is_current  
0    1985-03-22  2026-03-23  2026-03-23           0  
1  

C:\Users\Bayra\AppData\Local\Temp\ipykernel_11292\2887445164.py:43: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  dwh_conn.execute(
C:\Users\Bayra\AppData\Local\Temp\ipykernel_11292\2887445164.py:50: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  dwh_conn.execute(f"INSERT INTO {dwh_table} ({cols}) VALUES ({placeholders})", insert_values)
C:\Users\Bayra\AppData\Local\Temp\ipykernel_11292\2887445164.py:43: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  dwh_conn.execute(
C:\Users\Bayra\AppData\Local\Temp\ipykernel_11292\2887445164.py:50: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  dwh_conn.execute(f"INSE